In [2]:

import pandas as pd
df = pd.read_excel(r'C:\Users\Workstation\Documents\GitHub\welding-optuna-pytorch\comp test.xlsx')

In [5]:
df.rename(columns={"Unnamed: 0": "material"}, inplace=True)

In [9]:
df.columns

Index(['material', 'Material', 'EN-KN', 'EN Nm', 'GKI', 'C', 'Si', 'Mn', 'P',
       'S', 'N', 'Cr', 'Cu', 'Mo', 'Ni', 'V', 'Al', 'Co', 'Ti', 'Son', 'AISI',
       'Unnamed: 21', 'Unnamed: 22'],
      dtype='str')

In [11]:
dados = df[["material", "C", "Si", "Mn", "P", "S", "N", "Cr", "Mo", "Cu", "V", "Al", "Ni", "Ti", "Co"]]

In [12]:
dados

,material,C,Si,Mn,P,S,N,Cr,Mo,Cu,V,Al,Ni,Ti,Co
0,316,"≤ 0,07","≤ 1,00","≤ 2,00","≤ 0,045","≤ 0,03","≤ 0,10","16,5 - 18,5","2,00 - 2,50",-,-,NaN,"10,0 - 13,0",-,NaN
1,304,"≤ 0,07","≤ 1,00","≤ 2,00","≤ 0,045","≤ 0,03","≤ 0,10","17,5 - 19,5",-,-,-,NaN,"8,0 - 10,5",-,NaN
2,duplex 2205,"≤ 0,03",≤ 1,"≤ 2,0","≤ 0,03",-,"0,08-0,2",21 - 23,"2,5-3,5-",-,-,NaN,"4,5 - 6,5",-,NaN
3,A106 GrA,"≤ 0,25","≥ 0,1","0,27-0,93","≤ 0,035","≤ 0,035",-,"≤ 0,4","≤ 0,15","≤ 0,4","≤ 0,08",NaN,"≤ 0,4",-,NaN
4,A106 GrB,"≤ 0,30","≥ 0,1","0,29-1,06","≤ 0,035","≤ 0,035",-,"≤ 0,4","≤ 0,15","≤ 0,4","≤ 0,08",NaN,"≤ 0,4",-,NaN
5,A106 GrB,"≤ 0,35","≥ 0,1","0,29-1,06","≤ 0,035","≤ 0,035",-,"≤ 0,4","≤ 0,15","≤ 0,4","≤ 0,08",NaN,"≤ 0,4",-,NaN
6,A333 Gr1,0.3,-,"0,4-1,06",0.025,0.025,-,-,-,-,-,-,-,NaN,-
7,A333 Gr3,0.19,"0,18-0,37","0,31-0,64",0.025,0.025,-,-,-,-,-,-,"3,18-3,82",NaN,-
8,A333 Gr4,0.12,"0,08-0,37","0,5-1,05",0.025,0.025,-,"0,44-1,01",-,"0,4-0,75",-,"0,04-0,3","0,47-0,98",NaN,-
9,A333 Gr6,0.3,"≥ 0,1","0,29-1,06",0.025,0.025,-,"≤ 0,3",0.12,"≤ 0,4",0.08,-,"≤ 0,4",NaN,-


In [ ]:
def calcular_overlap(intervalo_a, intervalo_b):
    maior_inicio = max(intervalo_a[0], intervalo_b[0])
    menor_fim = min(intervalo_a[1], intervalo_b[1])
    return max(0, menor_fim - maior_inicio)

In [15]:
import pandas as pd
import numpy as np

# Sua função original
def calcular_overlap(intervalo_a, intervalo_b):
    if not intervalo_a or not intervalo_b:
        return 0
    maior_inicio = max(intervalo_a[0], intervalo_b[0])
    menor_fim = min(intervalo_a[1], intervalo_b[1])
    return max(0, menor_fim - maior_inicio)

# Função para converter os textos do DataFrame em tuplas (inicio, fim)
def extrair_intervalo(valor):
    if pd.isna(valor) or valor == '-':
        return None
    
    # Substituir vírgula por ponto para conversão float
    valor_str = str(valor).replace(',', '.').strip()
    
    try:
        if '≤' in valor_str:
            num = float(valor_str.replace('≤', '').strip())
            return (0.0, num) # Assume 0 como base para "menor ou igual"
        elif '≥' in valor_str:
            num = float(valor_str.replace('≥', '').strip())
            return (num, float('inf')) # Assume infinito para "maior ou igual"
        elif '-' in valor_str:
            partes = valor_str.split('-')
            return (float(partes[0].strip()), float(partes[1].strip()))
        else:
            # Para valores exatos únicos (caso existam)
            num = float(valor_str)
            return (num, num)
    except:
        return None

In [16]:
# Convertendo a coluna 'Cr' para intervalos reais
df['Cr_intervalo'] = df['Cr'].apply(extrair_intervalo)

In [17]:
import pandas as pd
import numpy as np

# 1. Função para extrair os números do texto
def extrair_intervalo(valor):
    if pd.isna(valor) or valor == '-':
        return None
    
    # Remove espaços e troca vírgula por ponto
    valor_str = str(valor).replace(',', '.').replace(' ', '')
    
    try:
        if '≤' in valor_str:
            num = float(valor_str.replace('≤', ''))
            return (0.0, num) # Ex: "≤ 0,07" vira (0.0, 0.07)
        elif '≥' in valor_str:
            num = float(valor_str.replace('≥', ''))
            return (num, 9999.0) # Assume 9999 como um limite alto
        elif '-' in valor_str:
            partes = valor_str.split('-')
            partes = [p for p in partes if p] # Remove espaços vazios se houver erro de digitação
            return (float(partes[0]), float(partes[1])) # Ex: "16,5 - 18,5" vira (16.5, 18.5)
        else:
            num = float(valor_str)
            return (num, num)
    except:
        return None

# 2. Sua função de overlap (com uma pequena proteção para valores nulos)
def calcular_overlap(intervalo_a, intervalo_b):
    if not intervalo_a or not intervalo_b:
        return 0.0
    maior_inicio = max(intervalo_a[0], intervalo_b[0])
    menor_fim = min(intervalo_a[1], intervalo_b[1])
    return max(0.0, menor_fim - maior_inicio)

# ==========================================
# COMO APLICAR NO SEU DATAFRAME (exemplo com a coluna 'Cr'):
# Substitua 'df' pelo nome da sua variável do DataFrame

# Passo A: Criar uma coluna nova com os intervalos limpos (tuplas matemáticas)
df['Cr_limpo'] = df['Cr'].apply(extrair_intervalo)

# Passo B: Definir o intervalo que você quer comparar. 
# Digamos que você quer saber quanto cada material sobrepõe o intervalo de 17.0 a 18.0
intervalo_alvo = (17.0, 18.0)

# Passo C: Aplicar a função de overlap usando lambda
df['Cr_overlap'] = df['Cr_limpo'].apply(lambda linha: calcular_overlap(linha, intervalo_alvo))

# Visualizar o resultado das colunas de interesse
print(df[['material', 'Cr', 'Cr_limpo', 'Cr_overlap']])

       material           Cr      Cr_limpo  Cr_overlap
0           316  16,5 - 18,5  (16.5, 18.5)         1.0
1           304  17,5 - 19,5  (17.5, 19.5)         0.5
2   duplex 2205      21 - 23  (21.0, 23.0)         0.0
3      A106 GrA        ≤ 0,4    (0.0, 0.4)         0.0
4      A106 GrB        ≤ 0,4    (0.0, 0.4)         0.0
5      A106 GrB        ≤ 0,4    (0.0, 0.4)         0.0
6      A333 Gr1            -          None         0.0
7      A333 Gr3            -          None         0.0
8      A333 Gr4    0,44-1,01  (0.44, 1.01)         0.0
9      A333 Gr6        ≤ 0,3    (0.0, 0.3)         0.0
10     A333 Gr7            -          None         0.0
11     A333 Gr8            -          None         0.0
12     A333 Gr9            -          None         0.0
13    A333 Gr10       ≤ 0,15   (0.0, 0.15)         0.0
14    A333 Gr11       ≤ 0,50    (0.0, 0.5)         0.0


In [22]:
def calcular_similaridade_percentual(int_a, int_b):
    if not int_a or not int_b: return 0.0
    if int_a == int_b: return 1.0 # Intervalos idênticos = 100%
        
    inicio_inter = max(int_a[0], int_b[0])
    fim_inter = min(int_a[1], int_b[1])
    intersecao = max(0.0, fim_inter - inicio_inter)
    
    if intersecao == 0: return 0.0
        
    inicio_uniao = min(int_a[0], int_b[0])
    fim_uniao = max(int_a[1], int_b[1])
    uniao = max(0.0001, fim_uniao - inicio_uniao) 
    
    return intersecao / uniao

elementos = ['C', 'Si', 'Mn', 'P', 'S', 'N', 'Cr', 'Mo', 'Cu', 'V', 'Al', 'Ni', 'Ti', 'Co']

dados_limpo = dados.copy()
for col in elementos:
    dados_limpo[col + '_int'] = dados[col].apply(extrair_intervalo)

# 3. Criar Matriz: MATERIAL vs MATERIAL
materiais = dados['material'].tolist()
matriz_materiais = pd.DataFrame(index=materiais, columns=materiais)

for i in range(len(materiais)):
    for j in range(len(materiais)):
        valores_similaridade = []
        
        # Compara elemento por elemento entre o Material A (linha i) e o Material B (linha j)
        for col in elementos:
            int_a = dados_limpo.loc[i, col + '_int']
            int_b = dados_limpo.loc[j, col + '_int']
            
            # Só considera para a média se ambos os materiais tiverem esse elemento listado
            if int_a is not None and int_b is not None:
                valores_similaridade.append(calcular_similaridade_percentual(int_a, int_b))
        
        # Tira a média de todas as sobreposições químicas entre os dois materiais
        if valores_similaridade:
            matriz_materiais.iloc[i, j] = round(np.mean(valores_similaridade), 3)
        else:
            matriz_materiais.iloc[i, j] = 0.0

print("=== Matriz de Similaridade Média (Material vs Material) ===")
matriz_materiais

=== Matriz de Similaridade Média (Material vs Material) ===


,316,304,duplex 2205,A106 GrA,A106 GrB,A106 GrB,A333 Gr1,A333 Gr3,A333 Gr4,A333 Gr6,A333 Gr7,A333 Gr8,A333 Gr9,A333 Gr10,A333 Gr11
316,1.0,0.804,0.399,0.281,0.282,0.278,0.082,0.059,0.081,0.048,0.107,0.107,0.066,0.053,0.081
304,0.804,1.0,0.456,0.321,0.322,0.317,0.082,0.059,0.081,0.055,0.107,0.187,0.066,0.061,0.093
duplex 2205,0.399,0.456,1.0,0.187,0.192,0.19,0.11,0.071,0.094,0.055,0.128,0.128,0.082,0.061,0.093
A106 GrA,0.281,0.321,0.187,1.0,0.964,0.952,0.168,0.083,0.069,0.456,0.113,0.113,0.112,0.138,0.182
A106 GrB,0.282,0.322,0.192,0.964,1.0,0.986,0.214,0.071,0.089,0.475,0.096,0.096,0.143,0.138,0.174
A106 GrB,0.278,0.317,0.19,0.952,0.986,1.0,0.214,0.071,0.089,0.475,0.096,0.096,0.143,0.138,0.174
A333 Gr1,0.082,0.082,0.11,0.168,0.214,0.214,1.0,0.58,0.708,0.964,0.618,0.618,0.75,0.0,0.547
A333 Gr3,0.059,0.059,0.071,0.083,0.071,0.071,0.58,1.0,0.474,0.405,0.658,0.492,0.464,0.105,0.485
A333 Gr4,0.081,0.081,0.094,0.069,0.089,0.089,0.708,0.474,1.0,0.339,0.506,0.506,0.472,0.103,0.412
A333 Gr6,0.048,0.055,0.055,0.456,0.475,0.475,0.964,0.405,0.339,1.0,0.429,0.429,0.476,0.15,0.362


In [26]:


df = pd.read_csv("C:\\Users\\Workstation\\Documents\\GitHub\\welding-optuna-pytorch\\embeddings_filtered.csv")

materiais = df["material"].tolist()
colunas_vetor = [c for c in df.columns if c != "material"]

def calcular_similaridade_cosseno(vetor_a, vetor_b):
    vetor_a = np.array(vetor_a, dtype=float)
    vetor_b = np.array(vetor_b, dtype=float)

    produto_escalar = np.dot(vetor_a, vetor_b)
    norma_a = np.linalg.norm(vetor_a)
    norma_b = np.linalg.norm(vetor_b)

    if norma_a == 0 or norma_b == 0:
        return 0.0

    return produto_escalar / (norma_a * norma_b)

matriz_materiais = pd.DataFrame(index=materiais, columns=materiais, dtype=float)

for i in range(len(materiais)):
    for j in range(len(materiais)):
        vetor_a = df.loc[i, colunas_vetor]
        vetor_b = df.loc[j, colunas_vetor]

        similaridade = calcular_similaridade_cosseno(vetor_a, vetor_b)

        matriz_materiais.loc[materiais[i], materiais[j]] = similaridade

print(matriz_materiais.round(3))
 

               316    304  duplex 2205  A333 Gr6  A333 Gr8
316          1.000 -0.283        0.157     0.293     0.176
304         -0.283  1.000       -0.183    -0.177     0.360
duplex 2205  0.157 -0.183        1.000    -0.006     0.065
A333 Gr6     0.293 -0.177       -0.006     1.000    -0.197
A333 Gr8     0.176  0.360        0.065    -0.197     1.000


In [25]:
import pandas as pd
import numpy as np


df = pd.read_csv("C:\\Users\\Workstation\\Documents\\GitHub\\welding-optuna-pytorch\\embeddings_filtered.csv")

materiais = df["material"].tolist()
colunas_vetor = [c for c in df.columns if c != "material"]

def calcular_distancia_euclidiana(vetor_a, vetor_b):
    vetor_a = np.array(vetor_a, dtype=float)
    vetor_b = np.array(vetor_b, dtype=float)

    return np.sqrt(np.sum((vetor_a - vetor_b) ** 2))

# Criar Matriz: MATERIAL vs MATERIAL
matriz_materiais = pd.DataFrame(index=materiais, columns=materiais, dtype=float)

for i in range(len(materiais)):
    for j in range(len(materiais)):
        vetor_a = df.loc[i, colunas_vetor]
        vetor_b = df.loc[j, colunas_vetor]

        distancia = calcular_distancia_euclidiana(vetor_a, vetor_b)

        matriz_materiais.loc[materiais[i], materiais[j]] = distancia


print(matriz_materiais.round(3))

               316    304  duplex 2205  A106 GrA  A106 GrB  A333 Gr6  A333 Gr8
316          0.000  8.276        5.309     5.009     6.749     5.343     6.076
304          8.276  0.000        7.054     6.742     7.949     7.657     5.930
duplex 2205  5.309  7.054        0.000     3.578     5.857     5.457     5.656
A106 GrA     5.009  6.742        3.578     0.000     5.013     6.204     5.956
A106 GrB     6.749  7.949        5.857     5.013     0.000     7.327     7.231
A333 Gr6     5.343  7.657        5.457     6.204     7.327     0.000     7.027
A333 Gr8     6.076  5.930        5.656     5.956     7.231     7.027     0.000
